# Preprocessing Spectra Data with Sirius
- combine annotated Spectra from study  with Spectra annotaded from Sirius

In [ ]:
import pandas as pd
from pyteomics import mgf
import re
import numpy as np
import json
import pickle    
from scipy.spatial.distance import pdist, squareform


In [ ]:
spectra = list(mgf.read("../../data/medical/RFA MSMS.mgf"))
print(f"Number of spectra: {len(spectra)}")

In [ ]:
df_spectra =pd.DataFrame(spectra)
print(df_spectra.columns)
df_spectra.head()

## 1. Sirius ran externally v6.10
- computation time ca. 6h

In [ ]:
sirius_results = pd.read_csv("../../data/medical/summary_run2/formula_identifications.tsv", sep="\t")
print(sirius_results.columns)
print(f"Number of sirius_results: {len(sirius_results)}")
sirius_results[:12]

In [ ]:
#adducts.iloc[0:5]['mappingFeatureId']
def extract_id_from_mapping_feature_id(mapping_feature_id):
    id = re.search(r'_UNKNOWN_(.*)', mapping_feature_id, flags=re.IGNORECASE)
    return id.group(1) if id else None
sirius_results['id'] = sirius_results['mappingFeatureId'].apply(extract_id_from_mapping_feature_id)
sirius_results.iloc[0:12]


In [ ]:
adducts = sirius_results['adduct'].unique()
print(f"Unique adducts: {adducts}")


In [ ]:
adductdiff = 0
counter = 0
adducts = dict()
for i, id in enumerate(sirius_results['id']):
    if id not in adducts:
        adducts[id] = sirius_results['adduct'][i]
        counter += 1
    elif adducts[id] != sirius_results['adduct'][i]:
        adductdiff += 1
        print(f"Adduct difference for id {id}: {adducts[id]} vs {sirius_results['adduct'][i]}")
print(f"Total ids that are similar but have different adducts: {adductdiff}")
print(f"Total unique ids: {counter}")


## 2. Neutral mass
- based on sirius adducts

In [ ]:
DEFAULT_INSTRUMENT = "Orbitrap"
DEFAULT_COLLISION_ENERGY = 50.0
DEFAULT_SIMULATION_CHALLENGE = False

def retention_time(string):
    rt = re.search(r'\(([\d\.]+)_', string)
    return float(rt.group(1)) if rt else None
def adduct_mass(string):
    if string == '[M + H]+':
        return 1.007276
    elif string == '[M + K]+':
        return 38.963158
    elif string == '[M + Na]+':
        return 22.989218
    else:
        raise AttributeError(f"Unknown adduct: {string}")

rows = []
counter = 0
big = 0
# create a json file
for inex, spec in df_spectra.iterrows():
    ID = spec["params"]["title"]
    new_ID = ID.replace("Unknown (", "")
    new_ID = new_ID.replace(")", "")
    rt = retention_time(ID)
    #print(f" ID {ID} has retention time {rt}")
    if ID not in adducts:
        print(f"ID {ID} not found in adducts, skipping")
        counter += 1
        continue
    adduct = adducts[ID]
    add_mass = adduct_mass(adduct)
    pepmass = spec["params"]["pepmass"][0]
    #print(f"ID {ID} has pepmass {pepmass}")
    neutral_mass = pepmass - add_mass
    if neutral_mass > 1000:
        big += 1
        continue
    peaks_json = np.column_stack((spec['m/z array'], spec['intensity array'])).tolist()

    entry = {
            "identifier": new_ID,
            "retention_time": rt,
            "neutral_mass": neutral_mass,
            "precursor_formula": None,  # not available
            "precursor_mz": pepmass,
            "peaks_mz": [],
            "adduct": adduct,
            "instrument_type": DEFAULT_INSTRUMENT,
            "collision_energy": DEFAULT_COLLISION_ENERGY,
            "simulation_challenge": DEFAULT_SIMULATION_CHALLENGE,
            "peaks_json": peaks_json
        }
    rows.append(entry)
json.dump(rows, open("../../data/medical/RFA MSMS_mz.json", "w"), indent=4)
print(f"Number of annotated Spectra by sirius: {len(rows)}")
print(f"Number of spectra with neutral mass above 1000: {big}")
print(f"Number of spectra without adduct information: {counter}")

## 3. Combine Datasets into one
- combine m/z (sirius annotated) and n (study annotated) into one spectra dataset

In [ ]:
# Build combined dataset of m/z and n spectra
mz_dataset = json.load(open("../../data/medical/RFA MSMS_mz.json"))
n_dataset = json.load(open("../../data/medical/RFA MSMS_n.json"))
msms_dataset = mz_dataset + n_dataset
print(f"Total number of MS/MS spectra: {len(msms_dataset)}")
occured = set()
nonunique = set()
count = 0
for spectrum in msms_dataset:
    spectrum_id = spectrum['identifier']
    if spectrum_id in occured:
        #print(f"Feature {spectrum_id} has a matching MS/MS spectrum")
        count += 1
        nonunique.add(spectrum_id)
    occured.add(spectrum_id)
#print(f"Number of features that occur multiple times:: {len(occured)}")
print(f"Total number of features non unique: {count}")
print(f"First 10  feature IDs: {list(occured)[:10]}")
print(f"First 10 non unique feature IDs: {list(nonunique)[:10]}")

In [ ]:
#Assign spectra with similar IDs an identifier based on occurences
identifier_counts = {}
new_msms_dataset = []
for spectrum in msms_dataset:
    spectrum_id = spectrum['identifier']
    if spectrum_id not in identifier_counts:
        identifier_counts[spectrum_id] = 1
    else:
        identifier_counts[spectrum_id] += 1
    new_id = f"{spectrum_id}_{identifier_counts[spectrum_id]}"
    new_spectrum = spectrum.copy()
    new_spectrum['identifier'] = new_id
    new_msms_dataset.append(new_spectrum)    

json.dump(new_msms_dataset, open("../../data/medical/RFA_MSMS_full.json", "w"), indent=4)